# 07 · Sync robusto diário com seleção manual

Por padrão, nada é baixado sem você selecionar. Arquivos já presentes no SSD são pulados; use `FORCE_RESYNC = True` quando quiser substituir uma cópia local.


In [ ]:
import sys
from pathlib import Path
import subprocess

SCRIPTS_DIR = Path("/kaggle/working/scripts")
sys.path.insert(0, str(SCRIPTS_DIR))
DATASET = "automamermaid/comfydocs"
TARGET_DIR = Path("/kaggle/working/ComfyUI/models")

from kaggle_sync import get_dataset_files, filter_dataset_files, sync_dataset_to_local, MODEL_CATEGORIES

def choose_dataset_files(dataset=DATASET, preselected_categories=None):
    """Abre uma caixa de seleção no notebook para escolher os arquivos exatos."""
    files = get_dataset_files(dataset)
    if not files:
        raise RuntimeError("O Dataset não possui arquivos ou não pôde ser listado.")

    candidates = filter_dataset_files(files, categories=preselected_categories)
    if not candidates:
        candidates = files

    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output

        options = [(f"{Path(f).name}  [{f}]", f) for f in candidates]
        selector = widgets.SelectMultiple(
            options=options,
            rows=min(18, max(5, len(options))),
            description="Modelos:",
            layout=widgets.Layout(width="100%", height="420px"),
        )
        select_all = widgets.Button(description="Selecionar todos")
        clear_all = widgets.Button(description="Limpar")
        confirm = widgets.Button(description="Confirmar seleção", button_style="success")
        output = widgets.Output()

        def all_click(_): selector.value = tuple(candidates)
        def clear_click(_): selector.value = tuple()
        def confirm_click(_):
            with output:
                clear_output(wait=True)
                chosen = list(selector.value)
                if not chosen:
                    print("Nenhum arquivo selecionado.")
                else:
                    print("Selecionados:")
                    for item in chosen: print("  -", item)
                    print(f"Total: {len(chosen)} arquivo(s)")

        select_all.on_click(all_click)
        clear_all.on_click(clear_click)
        confirm.on_click(confirm_click)
        display(widgets.HTML(f"<b>{dataset}</b> · {len(files)} arquivo(s) disponíveis · {len(candidates)} candidato(s)"))
        display(widgets.HBox([select_all, clear_all, confirm]))
        display(selector, output)

        # A execução da célula continua após a seleção. Aguarde a interface e rode a próxima célula.
        return selector
    except ImportError:
        print("ipywidgets não disponível. Use a lista abaixo e informe os paths manualmente.")
        for i, f in enumerate(candidates, 1): print(f"{i:03d}: {f}")
        raw = input("Números separados por vírgula: ").strip()
        indexes = [int(x)-1 for x in raw.split(',') if x.strip().isdigit()]
        return [candidates[i] for i in indexes if 0 <= i < len(candidates)]


In [ ]:
# Filtro opcional para reduzir a lista.
CATEGORIES = ["checkpoints", "diffusion_models", "loras", "vae", "text_encoders", "clip", "controlnet", "upscale_models", "video_models", "embeddings"]
selector = choose_dataset_files(DATASET, preselected_categories=CATEGORIES)


In [ ]:
if hasattr(selector, "value"):
    SELECTED_FILES = list(selector.value)
else:
    SELECTED_FILES = list(selector)

FORCE_RESYNC = False
if not SELECTED_FILES:
    raise ValueError("Nenhum arquivo selecionado.")

stats = sync_dataset_to_local(
    dataset=DATASET,
    target_dir=TARGET_DIR,
    selected_files=SELECTED_FILES,
    force=FORCE_RESYNC,
)

print("\nSYNC DIÁRIO CONCLUÍDO")
for d in stats["details"]:
    print(f"- {d['status']}: {d['path']}")
